# PLIER BP - Timing (3 runs)

💡 **Environment:** `clamp-analyses`  

In [1]:
library(here)
library(PLIER)
library(CLAMP)

source(here("config.R"))
set.seed(config$GTEx$RANDOM_SVD_SEED)

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses

Loading required package: RColorBrewer

Loading required package: gplots


---------------------
gplots 3.3.0 loaded:
  * Use citation('gplots') for citation info.
  * Homepage: https://talgalili.github.io/gplots/
  * Report issues: https://github.com/talgalili/gplots/issues
  * Ask questions: https://stackoverflow.com/questions/tagged/gplots
  * Suppress this message with: suppressPackageStartupMessages(library(gplots))
---------------------



Attaching package: ‘gplots’


The following object is masked from ‘package:stats’:

    lowess


Loading required package: pheatmap

Loading required package: glmnet

Loading required package: Matrix

Loaded glmnet 4.1-10

Loading required package: knitr

Loading required package: rsvd

Loading required package: qvalue


Attaching package: ‘CLAMP’


The following object is masked from ‘package:PLIER’:

    num.pc




In [2]:
input_dir <- config$GTEx$OUTPUT_DIR
output_dir <- here("output/model_performance/gtex")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

N_RUNS <- 3

In [3]:
gtex_fbm_filt <- readRDS(file.path(input_dir, "gtex_fbm_filt.rds"))
gtex_svdRes <- readRDS(file.path(input_dir, "gtex_svdRes.rds"))
CLAMP_K_gtex <- readRDS(file.path(input_dir, "CLAMP_K_gtex.rds"))
gtex_genes <- readRDS(file.path(input_dir, "gtex_genes.rds"))

In [4]:
gtex_gmtList <- list(
  BP = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2025")
)

for(lib in names(gtex_gmtList)) {
  names(gtex_gmtList[[lib]]) <- paste0(lib, "_", names(gtex_gmtList[[lib]]))
}

gtex_pathMat <- gmtListToSparseMat(gtex_gmtList)
gtex_matched <- getMatchedPathwayMat(gtex_pathMat, gtex_genes)
gtex_chatObj <- getChat(gtex_matched)

Auto-detected name: GO_Biological_Process_2025

Using cached file for GO_Biological_Process_2025

There are 12116 genes in the intersection between data and prior

Removing 2020 pathways

Inverting...

done



In [5]:
PLIER_times <- numeric(N_RUNS)

for (i in 1:N_RUNS) {
  cat("PLIER run", i, "of", N_RUNS, "\n")
  
  start_time <- Sys.time()
  
  gtex_plier <- PLIER::PLIER(
      gtex_fbm_filt[], 
      as.matrix(gtex_matched), 
      svdres = gtex_svdRes, 
      Chat = as.matrix(gtex_chatObj), 
      doCrossval = TRUE, 
      k = CLAMP_K_gtex
  )
  
  end_time <- Sys.time()
  PLIER_times[i] <- as.numeric(difftime(end_time, start_time, units = "mins"))
  cat("Run", i, "time:", PLIER_times[i], "minutes\n\n")
}

PLIER run 1 of 3 


Removing 0 pathways with too few genes



[1] 135.8334
[1] "L2 is set to 135.833443715207"
[1] "L1 is set to 67.9167218576034"


errorY (SVD based:best possible) = 0.3614

New L3 is 0.000158461325115751

New L3 is 0.00012340980408668

New L3 is 0.00012340980408668

New L3 is 0.000108908769855066

New L3 is 0.00012340980408668

New L3 is 0.00012340980408668

New L3 is 0.00012340980408668

New L3 is 0.000139841628594101

New L3 is 0.00012340980408668

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

converged at  iteration 185 Bdiff is not decreasing

There are 155  LVs with AUC>0.70



Run 1 time: 477.7285 minutes

PLIER run 2 of 3 


...

Run 2 time: 409.8696 minutes

PLIER run 3 of 3 


Removing 0 pathways with too few genes



[1] 135.8334
[1] "L2 is set to 135.833443715207"
[1] "L1 is set to 67.9167218576034"


errorY (SVD based:best possible) = 0.3614

New L3 is 0.000158461325115751

New L3 is 0.00012340980408668

New L3 is 0.00012340980408668

New L3 is 0.000108908769855066

New L3 is 0.00012340980408668

New L3 is 0.00012340980408668

New L3 is 0.00012340980408668

New L3 is 0.000139841628594101

New L3 is 0.00012340980408668

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

converged at  iteration 185 Bdiff is not decreasing

There are 155  LVs with AUC>0.70



Run 3 time: 478.34 minutes



In [6]:
PLIER_BP_time_minutes <- PLIER_times
names(PLIER_BP_time_minutes) <- paste0("run", 1:N_RUNS)
saveRDS(PLIER_BP_time_minutes, file.path(output_dir, "PLIER_BP_time_minutes.rds"))
cat("PLIER BP times:", PLIER_BP_time_minutes, "minutes\n")

PLIER BP times: 477.7285 409.8696 478.34 minutes
